# Tratamento e qualidade dos dados — Camada Silver

A camada Bronze preserva os dados recebidos da fonte. Este notebook prepara as oito tabelas para uso analítico, padronizando os campos, convertendo tipos e verificando problemas que possam afetar os indicadores da camada Gold.

## Objetivos do tratamento

- Converter datas, valores monetários e quantidades para tipos adequados.
- Preservar identificadores e prefixos de CEP como texto.
- Remover espaços externos dos campos estruturados e transformar textos vazios em valores nulos.
- Preservar o conteúdo dos comentários preenchidos.
- Identificar falhas de conversão, valores fora dos domínios esperados, inconsistências temporais e valores extremos.
- Salvar as tabelas tratadas em formato Delta no esquema `olist_mvp.silver`.

## Critérios de preservação

Valores ausentes não serão preenchidos por suposição. Datas, medidas e comentários desconhecidos permanecerão nulos. Produtos sem categoria ou sem tradução correspondente também serão mantidos.

As verificações realizadas na Bronze não identificaram duplicatas exatas. Nas avaliações, a combinação `review_id` e `order_id` apresentou unicidade, embora `review_id` isoladamente tenha repetições. Por isso, não será aplicada uma remoção indiscriminada de avaliações.

Valores extremos serão investigados como possíveis ocorrências atípicas, sem exclusão automática. Um preço elevado, por exemplo, pode representar uma venda legítima.

## Rastreabilidade e uso analítico

Os nomes originais das tabelas e colunas serão preservados, assim como os metadados de origem. Serão acrescentados o momento do processamento na Silver e a identificação das colunas com falha de conversão.

As verificações de qualidade serão apresentadas em resumos acompanhados de interpretação. Os registros serão preservados na Silver; filtros específicos serão aplicados na Gold conforme os requisitos de cada indicador.

A avaliação de qualidade verifica a consistência e a plausibilidade dos dados disponíveis. Sem uma fonte externa de conferência, ela não comprova a exatidão das transações reais.

In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS olist_mvp.silver")

# Colunas não listadas mantêm seus tipos da Bronze.
tipos_por_tabela = {
    "customers": {},
    "orders": {
        "order_purchase_timestamp": "timestamp_ntz",
        "order_approved_at": "timestamp_ntz",
        "order_delivered_carrier_date": "timestamp_ntz",
        "order_delivered_customer_date": "timestamp_ntz",
        "order_estimated_delivery_date": "timestamp_ntz",
    },
    "order_items": {
        "order_item_id": "int",
        "shipping_limit_date": "timestamp_ntz",
        "price": "decimal(18,2)",
        "freight_value": "decimal(18,2)",
    },
    "payments": {
        "payment_sequential": "int",
        "payment_installments": "int",
        "payment_value": "decimal(18,2)",
    },
    "reviews": {
        "review_score": "int",
        "review_creation_date": "timestamp_ntz",
        "review_answer_timestamp": "timestamp_ntz",
    },
    "products": {
        "product_name_lenght": "int",
        "product_description_lenght": "int",
        "product_photos_qty": "int",
        "product_weight_g": "decimal(18,3)",
        "product_length_cm": "decimal(18,3)",
        "product_height_cm": "decimal(18,3)",
        "product_width_cm": "decimal(18,3)",
    },
    "sellers": {},
    "category_translation": {},
}

campos_texto_livre = {
    "review_comment_title",
    "review_comment_message",
}

In [0]:
def preparar_tabela_silver(origem, conversoes):
    expressoes = []

    for coluna, tipo in origem.dtypes:
        valor = F.col(coluna)

        if tipo == "string" and not coluna.startswith("_"):
            texto = (
                valor
                if coluna in campos_texto_livre
                else F.trim(valor)
            )

            valor = F.when(
                F.trim(valor) == "",
                F.lit(None).cast("string")
            ).otherwise(texto)

        expressoes.append(valor.alias(coluna))

    dados = origem.select(*expressoes)

    # Registra valores preenchidos que não podem ser convertidos.
    falhas = [
        F.when(
            F.col(coluna).isNotNull()
            & F.expr(f"try_cast(`{coluna}` AS {tipo})").isNull(),
            F.lit(coluna)
        )
        for coluna, tipo in conversoes.items()
    ]

    dados = dados.withColumn(
        "_colunas_com_falha_conversao",
        F.filter(F.array(*falhas), lambda valor: valor.isNotNull())
        if falhas
        else F.expr("CAST(array() AS ARRAY<STRING>)")
    )

    dados = dados.select(*[
        (
            F.expr(f"try_cast(`{coluna}` AS {conversoes[coluna]})")
            if coluna in conversoes
            else F.col(coluna)
        ).alias(coluna)
        for coluna in dados.columns
    ])

    for coluna in ["customer_state", "seller_state"]:
        if coluna in dados.columns:
            dados = dados.withColumn(coluna, F.upper(F.col(coluna)))

    return dados.withColumn("_silver_timestamp", F.current_timestamp())


tabelas_preparadas = {
    tabela: preparar_tabela_silver(
        spark.table(f"olist_mvp.bronze.{tabela}"),
        conversoes
    )
    for tabela, conversoes in tipos_por_tabela.items()
}

## Padronização e conversão dos dados

O tratamento remove espaços externos dos campos textuais estruturados e converte textos vazios ou compostos apenas por espaços em valores nulos. Comentários preenchidos são preservados, mantendo seu conteúdo original.

Identificadores e prefixos de CEP permanecem como texto, pois representam códigos, não medidas. As siglas de unidades federativas são convertidas para maiúsculas.

Datas são convertidas para `timestamp_ntz`, sem conversão de fuso horário. Valores monetários utilizam duas casas decimais; pesos e dimensões, três. Quantidades e notas são convertidas para números inteiros.

Quando um valor preenchido não pode ser convertido para o tipo definido, o resultado torna-se nulo e o nome da coluna é registrado em `_colunas_com_falha_conversao`. Essa identificação permite distinguir falhas de conversão de ausências já existentes. O valor original permanece disponível na Bronze.

Nesta etapa, nenhuma linha é removida e nenhum valor ausente é preenchido por suposição. A etapa seguinte grava as tabelas na Silver e compara suas quantidades de registros com as da Bronze.

In [0]:
resumo_gravacao = []

for tabela, dados in tabelas_preparadas.items():
    destino = f"olist_mvp.silver.{tabela}"
    total_bronze = spark.table(f"olist_mvp.bronze.{tabela}").count()

    (
        dados.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(destino)
    )

    gravados = spark.table(destino)

    contagens = gravados.agg(
        F.count("*").alias("total_linhas"),
        F.coalesce(
            F.sum(
                F.when(
                    F.size("_colunas_com_falha_conversao") > 0, 1
                ).otherwise(0)
            ),
            F.lit(0)
        ).alias("linhas_com_falha")
    ).first()

    total_silver = contagens["total_linhas"]

    if total_bronze != total_silver:
        raise ValueError(
            f"Quantidade de linhas divergente em {tabela}: "
            f"Bronze={total_bronze}, Silver={total_silver}"
        )

    resumo_gravacao.append((
        tabela,
        total_bronze,
        total_silver,
        contagens["linhas_com_falha"]
    ))

display(spark.createDataFrame(
    resumo_gravacao,
    """
    tabela string,
    linhas_bronze long,
    linhas_silver long,
    linhas_com_falha_conversao long
    """
))

tabela,linhas_bronze,linhas_silver,linhas_com_falha_conversao
customers,99441,99441,0
orders,99441,99441,0
order_items,112650,112650,0
payments,103886,103886,0
reviews,99224,99224,0
products,32951,32951,0
sellers,3095,3095,0
category_translation,71,71,0


## Resultado da gravação e conversão

As oito tabelas foram salvas na camada Silver com as mesmas quantidades de linhas da Bronze. Não foram identificadas falhas de conversão nos campos submetidos à tipagem.

Esse resultado confirma a preservação da quantidade de registros e a conversibilidade dos valores preenchidos. Entretanto, uma conversão bem-sucedida não garante que o conteúdo seja válido: uma data pode estar corretamente formatada e ainda representar uma entrega anterior à compra.

A próxima etapa verifica os domínios dos campos e a consistência temporal. Valores ausentes, já examinados na Bronze, permanecem preservados e devem ser considerados conforme os requisitos de cada análise.

In [0]:
from pyspark.sql import functions as F

def verificar_regras(tabela, regras):
    dados = spark.table(f"olist_mvp.silver.{tabela}")

    contagens = dados.agg(
        F.count("*").alias("total"),
        *[
            F.coalesce(
                F.sum(F.when(condicao, 1).otherwise(0)),
                F.lit(0)
            ).alias(f"regra_{indice}")
            for indice, condicao in enumerate(regras.values())
        ]
    ).first()

    total = contagens["total"]

    return [
        (
            tabela,
            descricao,
            contagens[f"regra_{indice}"],
            round(100 * contagens[f"regra_{indice}"] / total, 2)
            if total else 0.0
        )
        for indice, descricao in enumerate(regras)
    ]


regras_datas_pedidos = {
    "Aprovação anterior à compra":
        F.col("order_approved_at") < F.col("order_purchase_timestamp"),
    "Envio anterior à compra":
        F.col("order_delivered_carrier_date")
        < F.col("order_purchase_timestamp"),
    "Entrega anterior ao envio":
        F.col("order_delivered_customer_date")
        < F.col("order_delivered_carrier_date"),
    "Entrega anterior à compra":
        F.col("order_delivered_customer_date")
        < F.col("order_purchase_timestamp"),
    "Previsão de entrega anterior à compra":
        F.col("order_estimated_delivery_date")
        < F.col("order_purchase_timestamp"),
    "Pedido entregue sem data de entrega":
        (F.col("order_status") == "delivered")
        & F.col("order_delivered_customer_date").isNull(),
}

regras_datas_avaliacoes = {
    "Resposta anterior à criação da avaliação":
        F.col("review_answer_timestamp")
        < F.col("review_creation_date"),
}

resultados_datas = (
    verificar_regras("orders", regras_datas_pedidos)
    + verificar_regras("reviews", regras_datas_avaliacoes)
)

esquema_qualidade = """
    tabela string,
    verificacao string,
    ocorrencias long,
    percentual double
"""

display(
    spark.createDataFrame(resultados_datas, esquema_qualidade)
    .orderBy(F.desc("ocorrencias"), "tabela", "verificacao")
)

tabela,verificacao,ocorrencias,percentual
orders,Envio anterior à compra,166,0.17
orders,Entrega anterior ao envio,23,0.02
orders,Pedido entregue sem data de entrega,8,0.01
orders,Aprovação anterior à compra,0,0.0
orders,Entrega anterior à compra,0,0.0
orders,Previsão de entrega anterior à compra,0,0.0
reviews,Resposta anterior à criação da avaliação,0,0.0


## Consistência temporal: resultados e tratamento

Foram identificados 166 pedidos com envio anterior à compra (0,17%), 23 com entrega anterior ao envio (0,02%) e 8 com status de entregue sem data de entrega (0,01%). Os percentuais utilizam o total de pedidos da tabela `orders`.

As demais regras temporais testadas não apresentaram ocorrências. Comparações que dependem de datas ausentes não são avaliáveis e não devem ser interpretadas como confirmação de consistência.

As ocorrências podem envolver os mesmos pedidos. Portanto, a soma das contagens por regra não representa necessariamente a quantidade de pedidos distintos com problemas.

As datas foram preservadas na Silver, pois não há informação suficiente para corrigi-las com segurança. Na Gold, a análise de pontualidade utiliza apenas pedidos com as datas necessárias preenchidas e cronologicamente consistentes, evitando calcular indicadores sobre sequências inválidas.

Essas restrições são específicas da análise de entregas. Uma inconsistência temporal não implica, por si só, que o preço dos itens do pedido esteja incorreto.

In [0]:
from pyspark.sql import functions as F

# Critérios de plausibilidade dos valores numéricos.
regras_numericas = {
    "order_items": {
        "Preço menor ou igual a zero": F.col("price") <= 0,
        "Frete negativo": F.col("freight_value") < 0,
        "Número do item menor que um": F.col("order_item_id") < 1,
    },
    "payments": {
        "Valor de pagamento negativo": F.col("payment_value") < 0,
        "Valor de pagamento igual a zero": F.col("payment_value") == 0,
        "Quantidade de parcelas menor que um": (
            F.col("payment_installments") < 1
        ),
        "Sequência de pagamento menor que um": (
            F.col("payment_sequential") < 1
        ),
    },
    "reviews": {
        "Nota fora do intervalo de 1 a 5": (
            ~F.col("review_score").between(1, 5)
        ),
    },
    "products": {
        "Peso menor ou igual a zero": F.col("product_weight_g") <= 0,
        "Comprimento menor ou igual a zero": (
            F.col("product_length_cm") <= 0
        ),
        "Altura menor ou igual a zero": F.col("product_height_cm") <= 0,
        "Largura menor ou igual a zero": F.col("product_width_cm") <= 0,
        "Quantidade de fotos negativa": F.col("product_photos_qty") < 0,
        "Comprimento do nome negativo": F.col("product_name_lenght") < 0,
        "Comprimento da descrição negativo": (
            F.col("product_description_lenght") < 0
        ),
    },
}

resultados_numericos = []

for tabela, regras in regras_numericas.items():
    resultados_numericos.extend(verificar_regras(tabela, regras))

display(
    spark.createDataFrame(resultados_numericos, esquema_qualidade)
    .orderBy(F.desc("ocorrencias"), "tabela", "verificacao")
)

tabela,verificacao,ocorrencias,percentual
payments,Valor de pagamento igual a zero,9,0.01
products,Peso menor ou igual a zero,4,0.01
payments,Quantidade de parcelas menor que um,2,0.0
order_items,Frete negativo,0,0.0
order_items,Número do item menor que um,0,0.0
order_items,Preço menor ou igual a zero,0,0.0
payments,Sequência de pagamento menor que um,0,0.0
payments,Valor de pagamento negativo,0,0.0
products,Altura menor ou igual a zero,0,0.0
products,Comprimento da descrição negativo,0,0.0


## Consistência dos valores numéricos

Foram identificados nove registros de pagamento com valor zero,
dois com quantidade de parcelas inferior a um e quatro produtos
com peso menor ou igual a zero.

Esses registros foram preservados na camada Silver. Pagamentos
zerados e parcelamentos inferiores a um precisam de esclarecimento
sobre as regras da origem; não há informação suficiente para
corrigi-los. Pesos não positivos são inadequados para análises
que dependam dessa medida, como estimativas logísticas.

Não foram encontrados preços de itens não positivos, fretes ou
pagamentos negativos, nem notas preenchidas fora do intervalo
de 1 a 5. As demais verificações numéricas também não apresentaram
ocorrências.

Os percentuais utilizam o total de linhas da respectiva tabela.
O percentual dos dois registros com menos de uma parcela aparece
como 0,00% devido ao arredondamento, embora existam ocorrências.
As contagens de regras diferentes podem incluir os mesmos registros.

Essas verificações avaliam limites de plausibilidade. Valores
extremamente altos, mas positivos, exigem uma análise adicional
de valores atípicos.

In [0]:
from pyspark.sql import functions as F

situacoes_pedido = [
    "created", "approved", "invoiced", "processing",
    "shipped", "delivered", "unavailable", "canceled"
]

tipos_pagamento = [
    "credit_card", "boleto", "voucher", "debit_card", "not_defined"
]

ufs = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF",
    "ES", "GO", "MA", "MT", "MS", "MG", "PA",
    "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
    "RO", "RR", "SC", "SP", "SE", "TO"
]

# O prefixo pode ter menos de cinco dígitos na origem.
# A verificação avalia o formato, sem confirmar a existência do CEP.
padrao_prefixo_cep = r"^[0-9]{1,5}$"

regras_categoricas = {
    "orders": {
        "Situação do pedido fora do domínio esperado": (
            ~F.col("order_status").isin(situacoes_pedido)
        ),
    },
    "payments": {
        "Tipo de pagamento fora do domínio esperado": (
            ~F.col("payment_type").isin(tipos_pagamento)
        ),
        "Tipo de pagamento não definido": (
            F.col("payment_type") == "not_defined"
        ),
    },
    "customers": {
        "UF não reconhecida": (
            ~F.col("customer_state").isin(ufs)
        ),
        "Prefixo de CEP fora do padrão de um a cinco dígitos": (
            ~F.col("customer_zip_code_prefix").rlike(padrao_prefixo_cep)
        ),
    },
    "sellers": {
        "UF não reconhecida": (
            ~F.col("seller_state").isin(ufs)
        ),
        "Prefixo de CEP fora do padrão de um a cinco dígitos": (
            ~F.col("seller_zip_code_prefix").rlike(padrao_prefixo_cep)
        ),
    },
}

resultados_categoricos = []

for tabela, regras in regras_categoricas.items():
    resultados_categoricos.extend(verificar_regras(tabela, regras))

display(
    spark.createDataFrame(resultados_categoricos, esquema_qualidade)
    .orderBy(F.desc("ocorrencias"), "tabela", "verificacao")
)

tabela,verificacao,ocorrencias,percentual
payments,Tipo de pagamento não definido,3,0.0
customers,Prefixo de CEP fora do padrão de um a cinco dígitos,0,0.0
customers,UF não reconhecida,0,0.0
orders,Situação do pedido fora do domínio esperado,0,0.0
payments,Tipo de pagamento fora do domínio esperado,0,0.0
sellers,Prefixo de CEP fora do padrão de um a cinco dígitos,0,0.0
sellers,UF não reconhecida,0,0.0


## Consistência das categorias e dos formatos

As situações dos pedidos e os tipos de pagamento preenchidos
pertencem aos domínios definidos para a análise. Entretanto, três
registros apresentam o tipo de pagamento `not_defined`: um valor
previsto na origem, mas que não informa o meio de pagamento utilizado.

Esses registros foram preservados com a classificação original,
pois não há evidência suficiente para atribuir outro tipo de pagamento.
Seu percentual aparece como 0,00% devido ao arredondamento.

Todas as UFs preenchidas de clientes e vendedores foram reconhecidas.
Os prefixos de CEP preenchidos contêm de um a cinco dígitos, conforme
o critério adotado para a representação encontrada na origem.
Essa verificação confirma apenas o formato; não comprova a existência
do CEP nem sua correspondência com a cidade ou a UF.

Valores ausentes são tratados na análise de completude. Portanto,
a ausência de ocorrências nestas regras não significa que todos
os campos estejam preenchidos.

In [0]:
from pyspark.sql import functions as F

campos_para_outliers = {
    "order_items": ["price", "freight_value"],
    "payments": ["payment_value", "payment_installments"],
    "products": [
        "product_weight_g", "product_length_cm",
        "product_height_cm", "product_width_cm",
        "product_photos_qty", "product_name_lenght",
        "product_description_lenght"
    ],
}

resultados_outliers = []

for tabela, colunas in campos_para_outliers.items():
    dados = spark.table(f"olist_mvp.silver.{tabela}")

    # Quartis aproximados; valores nulos são desconsiderados.
    quartis = dados.approxQuantile(colunas, [0.25, 0.75], 0.001)

    for coluna, limites in zip(colunas, quartis):
        if not limites:
            resultados_outliers.append((
                tabela, coluna, 0, None, None, None, None,
                None, None, None, None
            ))
            continue

        primeiro_quartil, terceiro_quartil = limites
        intervalo = terceiro_quartil - primeiro_quartil
        limite_inferior = primeiro_quartil - 1.5 * intervalo
        limite_superior = terceiro_quartil + 1.5 * intervalo

        valor = F.col(coluna)
        fora_dos_limites = (
            (valor < limite_inferior) | (valor > limite_superior)
        )

        resumo = dados.agg(
            F.count(valor).alias("preenchidos"),
            F.min(valor).cast("double").alias("minimo"),
            F.max(valor).cast("double").alias("maximo"),
            F.sum(
                F.when(fora_dos_limites, 1).otherwise(0)
            ).alias("atipicos")
        ).first()

        resultados_outliers.append((
            tabela,
            coluna,
            resumo["preenchidos"],
            resumo["minimo"],
            primeiro_quartil,
            terceiro_quartil,
            resumo["maximo"],
            limite_inferior,
            limite_superior,
            resumo["atipicos"],
            round(100 * resumo["atipicos"] / resumo["preenchidos"], 2)
        ))

esquema_outliers = """
    tabela string,
    coluna string,
    valores_preenchidos long,
    minimo double,
    primeiro_quartil double,
    terceiro_quartil double,
    maximo double,
    limite_inferior double,
    limite_superior double,
    valores_atipicos long,
    percentual double
"""

display(
    spark.createDataFrame(resultados_outliers, esquema_outliers)
    .orderBy(F.desc("percentual"), "tabela", "coluna")
)

tabela,coluna,valores_preenchidos,minimo,primeiro_quartil,terceiro_quartil,maximo,limite_inferior,limite_superior,valores_atipicos,percentual
products,product_weight_g,32949,0.0,300.0,1875.0,40425.0,-2062.5,4237.5,4609,13.99
order_items,freight_value,112650,0.0,13.08,21.15,409.68,0.9750000000000032,33.254999999999995,12134,10.77
payments,payment_value,103886,0.0,56.78,171.77,13664.08,-115.70500000000001,344.255,7990,7.69
order_items,price,112650,0.85,39.9,134.9,6735.0,-102.6,277.4,8427,7.48
products,product_height_cm,32949,2.0,8.0,20.0,105.0,-10.0,38.0,2385,7.24
products,product_description_lenght,32341,4.0,339.0,971.0,3992.0,-609.0,1919.0,2082,6.44
payments,payment_installments,103886,0.0,1.0,4.0,24.0,-3.5,8.5,6313,6.08
products,product_length_cm,32949,7.0,18.0,38.0,105.0,-12.0,68.0,1380,4.19
products,product_width_cm,32949,6.0,15.0,30.0,118.0,-7.5,52.5,912,2.77
products,product_photos_qty,32341,1.0,1.0,3.0,20.0,-2.0,6.0,849,2.63


## Valores atípicos: interpretação

A análise utilizou limites de 1,5 vez o intervalo interquartil
(IQR), calculados com quartis aproximados. Os percentuais consideram
apenas os valores preenchidos de cada coluna.

O peso dos produtos apresentou a maior proporção de valores atípicos:
4.609 produtos (13,99%), com pesos acima de 4.237,5 gramas. O maior
peso registrado foi de 40.425 gramas. Como o catálogo reúne produtos
de diferentes categorias, esses valores podem refletir diferenças
legítimas entre mercadorias.

Nos itens dos pedidos, 10,77% dos valores de frete ficaram fora
dos limites estatísticos, tanto por valores baixos quanto elevados.
O critério também pode sinalizar fretes gratuitos, que não são
necessariamente incorretos.

Os valores de pagamento e os preços dos itens apresentaram,
respectivamente, 7,69% e 7,48% de valores atípicos. O maior pagamento
registrado foi de R$ 13.664,08, enquanto o maior preço de item foi
de R$ 6.735,00. Essas medidas possuem granularidades diferentes:
um pagamento não corresponde necessariamente a um único item.

Também foram identificados valores atípicos nas dimensões dos
produtos, no parcelamento, na quantidade de fotos e nos comprimentos
dos nomes e das descrições. A sinalização indica distância em relação
à distribuição observada, sem comprovar erro de cadastro.

Todos esses valores foram preservados. Uma exclusão automática
poderia retirar operações legítimas e distorcer os indicadores
de vendas. Como aprofundamento, os limites poderiam ser avaliados
por categoria de produto.

## Conclusão da camada Silver

As oito tabelas foram gravadas em formato Delta no esquema
`olist_mvp.silver`, mantendo a quantidade de registros da Bronze.
Não foram identificadas falhas nas conversões de tipos.

O tratamento padronizou espaços em campos estruturados e siglas
de UF, converteu campos numéricos e temporais e preservou os textos
livres das avaliações. Valores ausentes permaneceram nulos, e os
metadados permitem acompanhar a origem e o processamento dos dados.

As verificações identificaram inconsistências temporais, pagamentos
zerados, parcelamentos inferiores a um, pesos não positivos e tipos
de pagamento não definidos. Esses registros foram mantidos porque
não há evidência suficiente para corrigir os valores originais.

A camada Silver oferece dados estruturados e limitações documentadas.
Na Gold, cada indicador aplica seus critérios de elegibilidade,
como a exigência de datas disponíveis e cronologicamente consistentes
para analisar atrasos nas entregas.

As verificações avaliam a qualidade interna dos dados. Sem uma fonte
externa de conferência, não é possível comprovar sua exatidão em
relação às operações reais.